#### From Single Prompts → Controlled Multi‑Step Reasoning

Environment Setup (Baseline)

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")
    


OpenAI API Key exists and begins sk-proj-


In [3]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [4]:
from openai import OpenAI

In [5]:
client = OpenAI()

Prompting = Inference‑Time Programming

Key framing for practitioners:
Prompting is inference‑time control, not training.

Cell 1 — Free‑form prompt (no control)

In [6]:
def ask(prompt, temperature=0.7, max_tokens=200):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

#ask("Explain feature leakage.")

Prompt Anatomy (Engineering View)

Prompt = Execution Contract

###### Why Single-Prompts Break in Practice
Overloaded prompt (anti‑pattern)

In [9]:
bad_prompt = """
Analyze the dataset, identify issues, recommend features,
choose a model, explain evaluation, and give deployment advice.
"""

ask(bad_prompt, temperature=0.7, max_tokens=300)

'To provide a structured response for the analysis of a dataset along with recommendations for features, model selection, evaluation, and deployment, let’s assume we are working on a generic supervised learning project, such as predicting house prices based on various features. \n\n### Step 1: Analyze the Dataset\n\n#### 1.1 Data Inspection\n- **Overview**: Check the number of records and features, data types, and basic statistics (mean, median, mode, range, etc.).\n- **Missing Values**: Identify any missing data points and their patterns. Use visualizations like heatmaps to spot these gaps.\n- **Outliers**: Use box plots and scatter plots to identify outliers that might skew the analysis.\n- **Feature Distribution**: Examine the distribution of each feature using histograms and density plots.\n\n#### 1.2 Issues Identified\n- **Missing Values**: Significant portions of data may be missing.\n- **Outliers**: Certain features may have extreme values that could influence model performance.

###### Decomposition Pattern (Prompt Chaining Mental Model)
Decompose by responsibility:

1. Understand the problem
2. Analyze data / constraints
3. Decide
4. Produce output


This mirrors classic ML pipeline stages.

######  Step 1 Prompt — Problem Understanding
Problem interpretation prompt

In [10]:
step1_prompt = """
SYSTEM: You are a senior ML engineer.

TASK:
Restate the problem and assumptions from the description below.

INPUT:
We want to predict customer churn using historical transaction data.

OUTPUT:
- Problem restatement
- Explicit assumptions (bullets)
"""

step1 = ask(step1_prompt, temperature=0.2, max_tokens=150)
print(step1)

**Problem Restatement:**
The objective is to develop a predictive model that identifies customers who are likely to discontinue their relationship with the company, utilizing historical transaction data as the primary input for the analysis.

**Explicit Assumptions:**
- Historical transaction data contains relevant features that correlate with customer churn.
- The patterns observed in past customer behavior are indicative of future churn tendencies.
- The dataset is sufficiently large and representative of the overall customer base to ensure reliable predictions.
- Customer churn can be defined and measured consistently within the dataset.
- External factors influencing customer behavior (e.g., market trends, economic conditions) are either negligible or accounted for in the analysis.
- The model will be trained on a balanced dataset that includes both churned and retained customers


###### Step 2 Prompt — Risk & Constraint Analysis
Risk identification

In [11]:
step2_prompt = f"""
SYSTEM: You are an ML risk reviewer.

CONTEXT:
{step1}

TASK:
List potential risks in building this model.

CONSTRAINTS:
- Focus on data leakage, bias, and evaluation
- Max 5 bullets
"""

step2 = ask(step2_prompt, temperature=0.2, max_tokens=150)
print(step2)

1. **Data Leakage**: There is a risk of inadvertently including future information in the training dataset, such as features that are derived from post-churn behavior, which could lead to overly optimistic model performance during evaluation and poor generalization to real-world scenarios.

2. **Bias in Historical Data**: The model may inherit biases present in the historical transaction data, such as demographic or behavioral biases, which could result in unfair predictions for certain customer segments, potentially leading to discriminatory practices in customer retention efforts.

3. **Imbalanced Dataset**: If the dataset is not truly balanced despite assumptions, with a significantly higher number of retained customers compared to churned ones, the model may become biased towards predicting the majority class, leading to poor identification of actual


###### Step 3 Prompt — Recommendation Generation
Decision prompt

In [12]:
step3_prompt = f"""
SYSTEM: You are an ML solution architect.

CONTEXT:
Problem Summary:
{step1}

Identified Risks:
{step2}

TASK:
Recommend:
- Feature strategy
- Model family
- Validation approach

FORMAT:
Bulleted sections
"""

step3 = ask(step3_prompt, temperature=0.3, max_tokens=250)
print(step3)

### Feature Strategy
- **Feature Selection**:
  - Identify key features from historical transaction data such as frequency of purchases, average transaction value, recency of last purchase, and customer engagement metrics (e.g., website visits, customer service interactions).
  - Include demographic features (age, gender, location) to capture potential biases and customer segments.
  - Create derived features that capture trends over time, such as moving averages or changes in transaction behavior.

- **Feature Engineering**:
  - Develop features that indicate changes in customer behavior, such as a sudden drop in transaction frequency or value.
  - Implement time-based features to capture seasonality or specific events that may influence churn (e.g., holidays, promotions).
  - Use clustering techniques to segment customers based on transaction patterns and behaviors, which can help in understanding different churn dynamics.

- **Handling Missing Data**:
  - Employ imputation technique

###### Why We Avoid “Show Your Reasoning”

Important production principle

Do not ask for chain‑of‑thought explicitly.
Instead, ask for:

1. Intermediate artifacts
2. Summaries
3. Structured decisions


❌ What NOT to do

In [13]:
ask("Think step by step and show all your reasoning to solve this ML problem.")


'Sure! Please provide me with the specific machine learning (ML) problem you would like me to help you solve, and I will guide you through the step-by-step reasoning.'

✅ Correct pattern

In [14]:
ask("""
Solve the problem internally.
Return ONLY the final recommendations and key decision factors.
""", temperature=0.2)

'Sure! Please provide the details of the problem you would like me to solve.'

✅ You get clean outputs without reasoning leakage.

###### Functional Prompt Chaining (Reusable Code Pattern)
Prompt pipeline utility

In [15]:
def run_step(system, task, context=None, max_tokens=200, temperature=0.2):
    prompt = f"SYSTEM:\n{system}\n\n"
    if context:
        prompt += f"CONTEXT:\n{context}\n\n"
    prompt += f"TASK:\n{task}"
    return ask(prompt, temperature=temperature, max_tokens=max_tokens)


Use it like a pipeline

✅ This starts to look like inference‑time orchestration.

In [16]:
s1 = run_step(
    system="You are an ML analyst.",
    task="Restate the churn prediction problem and assumptions."
)

s2 = run_step(
    system="You are an ML risk auditor.",
    context=s1,
    task="List key ML and data risks."
)

s3 = run_step(
    system="You are a solution architect.",
    context=s1 + "\n" + s2,
    task="Recommend modeling and validation strategy."
)

print(s3)

To effectively address the churn prediction problem, a robust modeling and validation strategy is essential. Below is a recommended approach that encompasses data preparation, model selection, training, validation, and evaluation.

### 1. Data Preparation

#### a. Data Cleaning
- **Handle Missing Values**: Use imputation techniques (mean, median, mode, or predictive modeling) to fill in missing values or remove records with excessive missing data.
- **Correct Inaccuracies**: Implement data validation rules to identify and correct inaccuracies in the dataset.

#### b. Feature Engineering
- **Create Relevant Features**: Derive new features that may capture customer behavior more effectively (e.g., average transaction value, frequency of customer service interactions).
- **Feature Scaling**: Normalize or standardize numerical features to ensure they contribute equally to the model's performance.

#### c. Data Segmentation
- **Customer Segmentation**: Segment customers based on demographic

###### Prompt Chaining Failure Modes (Critical Section)
Common mistakes practitioners make

1. Passing too much context (token bloat)
2. Carrying forward hallucinations
3. No intermediate validation
4. No stop conditions

###### Validation gate

✅ Introduces quality gates in prompt chains.

In [17]:
validator_prompt = f"""
You are a reviewer.

INPUT:
{s3}

TASK:
Identify any assumptions that require data validation.
Return bullets only.
"""

ask(validator_prompt, temperature=0.1, max_tokens=120)

'- Assumption that missing values can be accurately imputed without introducing bias.\n- Assumption that the derived features will have a significant impact on predicting churn.\n- Assumption that the chosen segmentation criteria will effectively differentiate customer behavior.\n- Assumption that the data cleaning process will not remove critical information from the dataset.\n- Assumption that normalization or standardization will not distort the relationships between features.'

##### Mini‑Lab (Hands‑On for Participants)

In [19]:
lab_prompt = """
Create 3 hands-on labs for prompt chaining in ML workflows.
Each lab must include:
- Use case
- Naive single prompt
- Decomposed prompt steps
- Expected improvement

Each lab ≤ 8 lines.
"""

ask(lab_prompt, temperature=0.6, max_tokens=600)

'### Lab 1: Sentiment Analysis of Product Reviews\n\n- **Use Case:** Analyze customer reviews to determine overall sentiment.\n- **Naive Single Prompt:** "What is the sentiment of the following review: \'This product is amazing!\'"\n- **Decomposed Prompt Steps:** \n  1. "Identify the key phrases in the review."\n  2. "Classify each key phrase as positive, negative, or neutral."\n  3. "Summarize the overall sentiment based on the classifications."\n- **Expected Improvement:** Increased accuracy in sentiment classification by considering context and nuances in language.\n\n---\n\n### Lab 2: Automated Email Response Generation\n\n- **Use Case:** Automatically generate responses to customer inquiries.\n- **Naive Single Prompt:** "Write a response to this email: \'I have a question about my order.\'"\n- **Decomposed Prompt Steps:**\n  1. "Extract the main question from the email."\n  2. "Identify the relevant information needed to answer the question."\n  3. "Compose a polite and informativ

###### Module 2 Takeaways

1. One prompt ≠ one solution
2. Decompose prompts like ML pipelines
3. Chain prompts using artifacts, not raw reasoning
4. Validate intermediate steps
5. Prompt chaining is inference‑time orchestration